## Kick Analysis of correlation

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("Complete_Data_2025.csv", parse_dates=['date'], index_col='date', low_memory=False)
# Rename price column
df = df.rename(columns={'Deutschland/Luxemburg [€/MWh] Originalauflösungen': 'Price'})

# To numeric
df = df.apply(pd.to_numeric, errors='coerce')

# Calculate correlation
corr = (
    df.corrwith(df['Price'])
      .drop('Price')
      .dropna()
      .sort_values(ascending=False)
      .rename('correlation')
      .reset_index()
      .rename(columns={'index': 'columna'})
)

def correlation(r):
    r = abs(r)
    if r >= 0.7:  return 'Too strong'
    if r >= 0.5:  return 'Strong'
    if r >= 0.3:  return 'Moderate'
    if r >= 0.1:  return 'Low'
    return 'null'

corr['Level'] = corr['correlation'].apply(correlation)

pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:.4f}'.format)
print(corr.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import skew, kurtosis

diag = pd.DataFrame({
    'nan_count':     df.isnull().sum(),
    'nan_pct':       (df.isnull().sum() / len(df) * 100).round(2),
    'mean':          df.mean().round(2),
    'median':        df.median().round(2),
    'std':           df.std().round(2),
    'min':           df.min().round(2),
    'max':           df.max().round(2),
    'p25':           df.quantile(0.25).round(2),
    'p75':           df.quantile(0.75).round(2),
    'iqr':           (df.quantile(0.75) - df.quantile(0.25)).round(2),
    'n_outliers_z4': df.apply(lambda x: int((np.abs((x - x.mean()) / x.std()) > 4).sum())),
})
diag.index.name = 'columna'

# Exportar tabla
diag.to_csv("diagnostico_estadistico.csv")
print(diag.to_string())



## Apply Windsorisation

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
import plotly.graph_objects as go

original_price = df['Price'].copy()

# delete duplicates
data1 = df[~df.index.duplicated(keep='first')]

# Interpolation
data1 = data1.interpolate(method='time').ffill().bfill()

# Outliers windsorisation
def winsorize(df, z_thresh=4):
    df = df.copy()
    for col in df.columns:
        z = np.abs(zscore(df[col], nan_policy='omit'))
        inliers = df[col][z <= z_thresh]
        df[col] = df[col].clip(inliers.min(), inliers.max())
    return df

data_wind = winsorize(data1)
# guardar como CSV
data_wind.to_csv("data_test1.csv", index=True, encoding="utf-8")

In [ ]:
#Data Plotting

fig = go.Figure()
fig.add_trace(go.Scatter(x=original_price.index, y=original_price,
                         mode='lines', name='Price Original', line=dict(color='gray')))
fig.add_trace(go.Scatter(x=data_wind.index, y=data_wind['Price'],
                         mode='lines', name='Price Process', line=dict(color='green')))
# --- Eje secundario para coal y gas ---
fig.add_trace(go.Scatter(x=data_wind.index, y=data_wind['coal_EUR_t'],
                         mode='lines', name='Coal (€/t)', line=dict(color='brown'),
                         yaxis='y2'))
fig.add_trace(go.Scatter(x=data_wind.index, y=data_wind['gas_EUR_MWh'],
                         mode='lines', name='Gas (€/MWh)', line=dict(color='red'),
                         yaxis='y2'))
fig.update_layout(
    title="Price, Coal and Gas",
    xaxis_title="Date",
    height=600,
    yaxis=dict(title="Electricity price (€/MWh)"),
    yaxis2=dict(title="Coal (€/t) / Gas (€/MWh)", overlaying='y', side='right'),
    legend=dict(x=0, y=1)
)

fig.show()

In [ ]:
import plotly.graph_objects as go

# --- Media diaria ---
original_daily   = original_price.resample('D').mean()
price_daily      = data_wind['Price'].resample('D').mean()
pv_daily         = data_wind['PV'].resample('D').mean()
won_daily        = data_wind['WON'].resample('D').mean()

fig = go.Figure()

# --- Precio ---
fig.add_trace(go.Scatter(x=price_daily.index, y=price_daily,
                         mode='lines', name='Price Processed',
                         line=dict(color='green', width=1.5)))

# --- Solar y Eólica (eje secundario) ---
fig.add_trace(go.Scatter(x=pv_daily.index, y=pv_daily,
                         mode='lines', name='Solar PV (MWh)',
                         line=dict(color='orange', width=1.5),
                         yaxis='y2'))
fig.add_trace(go.Scatter(x=won_daily.index, y=won_daily,
                         mode='lines', name='Wind Onshore (MWh)',
                         line=dict(color='royalblue', width=1.5),
                         yaxis='y2'))

fig.update_layout(
    title="Electricity Price, Solar & Wind — Daily Average",
    xaxis=dict(title="Date", rangeslider=dict(visible=True),  # slider para zoom
               rangeselector=dict(buttons=[                    # botones de rango
                   dict(count=1,  label="1M", step="month", stepmode="backward"),
                   dict(count=6,  label="6M", step="month", stepmode="backward"),
                   dict(count=1,  label="1Y", step="year",  stepmode="backward"),
                   dict(step="all", label="All")
               ])),
    yaxis =dict(title="Electricity Price (€/MWh)"),
    yaxis2=dict(title="Generation (MWh)", overlaying='y', side='right'),
    legend=dict(x=0, y=1),
    height=600,
    template='plotly_white'
)

fig.show()

In [ ]:
import plotly.graph_objects as go

# --- Media diaria ---
original_daily   = original_price.resample('D').mean()
price_daily=data_wind['Price'].resample('D').mean()
wind_hassel=data_wind['wind_speed_100m_Hassel'].resample('D').mean()
wind_bremen=data_wind['wind_speed_100m_Bremen'].resample('D').mean()
wind_potsdam=data_wind['wind_speed_100m_Potsdam'].resample('D').mean()

fig = go.Figure()

# --- Precio ---
fig.add_trace(go.Scatter(x=price_daily.index, y=price_daily,
                         mode='lines', name='Price Processed',
                         line=dict(color='green', width=1.5)))

# --- Solar y Eólica (eje secundario) ---
fig.add_trace(go.Scatter(x=wind_hassel.index, y=wind_hassel,
                         mode='lines', name='wind_speed_100m_Hassel',
                         line=dict(color='orange', width=1.5),
                         yaxis='y2'))
fig.add_trace(go.Scatter(x=wind_bremen.index, y=wind_bremen,
                         mode='lines', name='wind_speed_100m_Bremen',
                         line=dict(color='royalblue', width=1.5),
                         yaxis='y2'))
fig.add_trace(go.Scatter(x=wind_potsdam.index, y=wind_potsdam,
                         mode='lines', name='wind_speed_100m_Potsdam',
                         line=dict(color='black', width=1.5),
                         yaxis='y2'))

fig.update_layout(
    title="Electricity Price, Solar & Wind — Daily Average",
    xaxis=dict(title="Date", rangeslider=dict(visible=True),  # slider para zoom
               rangeselector=dict(buttons=[                    # botones de rango
                   dict(count=1,  label="1M", step="month", stepmode="backward"),
                   dict(count=6,  label="6M", step="month", stepmode="backward"),
                   dict(count=1,  label="1Y", step="year",  stepmode="backward"),
                   dict(step="all", label="All")
               ])),
    yaxis =dict(title="Electricity Price (€/MWh)"),
    yaxis2=dict(title="Wind speed (m/s)", overlaying='y', side='right'),
    legend=dict(x=0, y=1),
    height=600,
    template='plotly_white'
)

fig.show()

